# Evaluation: measure and compare approaches on your test set

This notebook answers "how well does it work, and which variant is better?" on your labelled test set (made with `notebooks/labeling.ipynb`). For trying things by hand, use `notebooks/search.ipynb`.

**How it is measured**
- **Every labelled box is a query once** (leave-one-out). It is searched against all other boxes and the distractor images. Everything from the query's own photo is left out, so the object must be found in *other* photos.
- **Two levels of "correct", side by side:**
  - **item:** the result has the same label, i.e. the exact physical object.
  - **model:** the result has the same `group` (e.g. the same sofa model). Boxes without a group count as their own group, so this is never stricter than item level.
- **Metrics:**
  - `R@1` / `R@5`: the share of queries with a correct result at rank 1 / in the top 5.
  - `MRR`: the mean of 1/rank of the first correct result (1.0 = always first).
  - Columns starting with `model` are the model-level versions.

**Sections**
1. **Setup.**
2. **Test set.**
3. **Stage 1:** retrieval methods.
4. **Stage 2:** keypoint verification variants.
5. **Grounding:** what if the photos had no boxes?
6. **Per-label breakdown.**
7. **Failure gallery.**
8. **Latency.**
9. **Save the results.**

## 1. Setup

This cell imports the package and sets the knobs for this evaluation run:
- `TESTSET_DIR`: the labelled test set (default `data/testset/`, or the `VS_TESTSET` environment variable).
- `DINO_MODEL`, `DINO_RESOLUTION`, `RETRIEVAL_K`, `SPARSE_K`: as in the search notebook.
- `IOU_THRESHOLD`: when a region found by grounding counts as hitting the right box (section 5).
- `RUN_NAME`, `RESULTS_DIR`: where section 9 saves the tables (CSV).

In [ ]:
import logging
import os
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

from visual_search.cascade import Cascade
from visual_search.data import AnnotationSet, make_synthetic_dataset, validate_dataset
from visual_search.data.schema import parse_dataset
from visual_search.device import device_sync
from visual_search.eval import (
    GroundedJudge,
    comparison_table,
    default_judges,
    evaluate_leave_one_out,
    run_evaluation,
)
from visual_search.eval.inspect import (
    failure_grid,
    latency_table,
    localization_stats,
    per_label_table,
    save_tables,
    worst_queries,
)
from visual_search.matching import create_matcher
from visual_search.models.dinov2 import Dinov2Backbone
from visual_search.retrieval import POOLINGS, ColorHistogramRetriever, DinoRetriever, GroundedDinoRetriever, RandomRetriever
from visual_search.stages import SparseVerificationStage

logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")
pd.set_option("display.precision", 3)
ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
TESTSET_DIR = Path(os.environ.get("VS_TESTSET", ROOT / "data" / "testset"))
CACHE_DIR = ROOT / "cache"
RESULTS_DIR = ROOT / "results"
RUN_NAME = "baseline"

DINO_MODEL = "facebook/dinov2-small"
DINO_RESOLUTION = 448
RETRIEVAL_K = 100
SPARSE_K = 20
IOU_THRESHOLD = 0.3

## 2. Test set

This cell loads `annotations.json` and turns every box into a test item.

- **Overview:** the validator's report shows how many objects (labels) and distractors there are, and which labels have only one box. Those can't be evaluated as queries, but they still act as gallery entries.
- **Fallback:** without a test set, generated synthetic data is used (a smoke test only: it shows the notebook works, not which method is better). In that case section 5 is skipped, because it needs real annotated photos.

In [ ]:
if (TESTSET_DIR / "annotations.json").exists():
    testset = AnnotationSet.load(TESTSET_DIR)
    print(testset.summary())
    items = parse_dataset(testset.to_records())
else:
    testset = None
    print(f"No annotations.json in {TESTSET_DIR}; using synthetic smoke-test data.")
    items = parse_dataset(make_synthetic_dataset(n_instances=12, views_per_instance=3, n_distractors=40,
                                                 n_near_duplicates=3, seed=0))
print(validate_dataset(items).summary())
n_queries = sum(it.label is not None for it in items)
n_groups = len({it.group for it in items if it.group})
print(f"{n_queries} labelled boxes (queries), {len(items) - n_queries} distractors, {n_groups} groups")

backbone = Dinov2Backbone(DINO_MODEL, resolution=DINO_RESOLUTION)
sync = device_sync(backbone.device)
JUDGES = default_judges()  # {"item": exact label, "model": same group}

## 3. Stage 1: retrieval methods

This cell compares retrieval methods without any re-ranking:
- `random`: random order, the chance level.
- `color_hist`: a color histogram, which shows what plain color achieves.
- DINOv2 `cls`: the whole query image, ignoring the box.
- DINOv2 `crop_cls`: the query box cropped and described (the default).
- DINOv2 `masked_mean`: the patch features under the box.

Gallery boxes are described by their box crop. Features are cached in `cache/`, so only the first run is slow.

In the table, compare `R@1` (exact item) with `model R@1` (same model). The latency columns are per query, in ms.

In [ ]:
def stage1_reports():
    """Leave-one-out reports for the baselines and every DINOv2 pooling mode."""
    reports = [evaluate_leave_one_out(Cascade(RandomRetriever()), items, judges=JUDGES),
               evaluate_leave_one_out(Cascade(ColorHistogramRetriever()), items, judges=JUDGES)]
    for pooling in POOLINGS:
        retriever = DinoRetriever(backbone, pooling=pooling, cache_dir=CACHE_DIR)
        reports.append(evaluate_leave_one_out(Cascade(retriever, RETRIEVAL_K, sync=sync), items, judges=JUDGES))
    return reports


stage1 = stage1_reports()
stage1_table = comparison_table(stage1).drop(columns=["stage"])
stage1_table

## 4. Stage 2: keypoint verification variants

DINOv2 `crop_cls` retrieves the top `RETRIEVAL_K`. Then the top `SPARSE_K` are re-ranked by the number of geometrically confirmed keypoint matches, inside the query box and inside each gallery box.

Variants compared:
- **estimator:** OpenCV MAGSAC++ vs classic RANSAC;
- **geometric model:** homography `H` (flat objects), affine `A`, fundamental `F` (3D);
- **ablations:** no masks (whole images), plain SIFT instead of RootSIFT;
- **learned matchers:** ALIKED + LightGlue and XFeat + LighterGlue. They need `uv sync --extra sparse` and re-rank only the top 10, since they are slower per pair.

Geometric verification checks for the *same physical surface*. Expect it to help item level more than model level: two copies of a model differ in creases, wear and lighting.

In [ ]:
LEARNED = {}
for _name in ["aliked-lightglue", "xfeat-lighterglue"]:
    try:
        LEARNED[_name] = create_matcher(_name)
    except ImportError as exc:
        print(f"skipping {_name}: {exc}")


def stage2_configs():
    """(name, stages) pairs for the stage-2 comparison."""
    sparse = lambda **kw: SparseVerificationStage(top_k=SPARSE_K, cache_dir=CACHE_DIR, **kw)
    return [
        ("stage 1 only", []),
        ("sift H magsac", [sparse(matcher="sift", model_type="H", method="magsac")]),
        ("sift H ransac", [sparse(matcher="sift", model_type="H", method="ransac")]),
        ("sift A magsac", [sparse(matcher="sift", model_type="A")]),
        ("sift F magsac", [sparse(matcher="sift", model_type="F")]),
        ("sift H magsac, no mask", [sparse(matcher="sift", use_masks=False, name="sift-nomask")]),
        ("plain SIFT H magsac", [sparse(matcher=create_matcher("sift", root_sift=False), name="sift-plain")]),
    ] + [(f"{name} H magsac (top-10)", [SparseVerificationStage(m, top_k=10, cache_dir=CACHE_DIR)])
         for name, m in LEARNED.items()]


stage2 = []
for label, stages in stage2_configs():
    cascade = Cascade(DinoRetriever(backbone, pooling="crop_cls", cache_dir=CACHE_DIR), RETRIEVAL_K, stages, sync=sync)
    stage2.append(evaluate_leave_one_out(cascade, items, name=label, judges=JUDGES))
stage2_table = comparison_table(stage2).drop(columns=["stage"])
stage2_table

## 5. Grounding: what if the photos had no boxes?

Your boxes tell the system where every object is. In real use (`search.ipynb`, section 4) the gallery is plain photos, and the system must find the object itself. This cell simulates that: **the boxes are hidden from the gallery, which becomes your test-set photos**. The query boxes stay.

Three variants:
- **annotated boxes:** the gallery is your boxes. This is the upper bound: what perfect grounding would give.
- **DINOv2 grounding:** `GroundedDinoRetriever` finds regions in each photo. A region is correct when it overlaps a box of the right label with IoU ≥ `IOU_THRESHOLD`.
- **whole images:** no grounding. Each photo is described as a whole and counts as correct when it contains the object, regardless of where.

All variants use SIFT verification afterwards (inside the box, region or whole image). The second table shows how well the found regions overlap your boxes:
- `median_iou`: the median overlap;
- `share_iou_0.3` / `share_iou_0.5`: the share of queries whose first region in a correct photo overlaps the right box by at least 0.3 / 0.5.

In [ ]:
if testset is None:
    print("Skipped: needs a labelled test set with real photos.")
    grounding_table, localization_table = pd.DataFrame(), pd.DataFrame()
else:
    photos = parse_dataset(testset.image_records())
    boxes = testset.boxes_by_image()
    queries = [it for it in items if it.label is not None]
    grounded_judges = {"item": GroundedJudge(boxes, IOU_THRESHOLD), "model": GroundedJudge(boxes, IOU_THRESHOLD, level="model")}
    verify = lambda: [SparseVerificationStage("sift", top_k=SPARSE_K, cache_dir=CACHE_DIR)]

    grounded_retriever = GroundedDinoRetriever(backbone, cache_dir=CACHE_DIR)
    grounding = [
        evaluate_leave_one_out(Cascade(DinoRetriever(backbone, "crop_cls", CACHE_DIR), RETRIEVAL_K, verify(), sync=sync),
                               items, name="annotated boxes", judges=JUDGES),
        run_evaluation(Cascade(grounded_retriever, RETRIEVAL_K, verify(), sync=sync), queries, photos,
                       grounded_judges, name="DINOv2 grounding"),
        run_evaluation(Cascade(DinoRetriever(backbone, "crop_cls", CACHE_DIR), RETRIEVAL_K, verify(), sync=sync),
                       queries, photos, grounded_judges, name="whole images"),
    ]
    grounding_table = pd.concat({"stage 1": comparison_table(grounding, stage="retrieval"),
                                 "after SIFT": comparison_table(grounding)}, names=["after"]).drop(columns=["stage"])
    localization_table = pd.DataFrame([localization_stats(grounded_retriever, queries, photos, grounded_judges["item"])],
                                      index=["DINOv2 grounding"])
    display(grounding_table)
    display(localization_table)

## 6. Per-label breakdown

Which objects are easy and which are hard? This cell shows one row per label, for the default pipeline (DINOv2 `crop_cls` → SIFT H MAGSAC++, item level), hardest first:
- `queries`: how many boxes of that label were used as queries;
- `R@1`, `R@5`, `MRR`: as above, but for this label only;
- `median rank`: the typical rank of the first correct result;
- `not found`: queries whose correct boxes weren't in the result list at all.

Labels at the top are worth a look in the failure gallery, or more photos.

In [ ]:
default_report = next(r for r in stage2 if r.name == "sift H magsac")
labels_table = per_label_table(default_report)
labels_table

## 7. Failure gallery

This cell draws the worst queries of the default pipeline: the query box (left), then the top 5 per stage, each with a green ✓ or red ✗ frame at item level. Use it to see *why* things fail, for example:
- a look-alike object ranked first (same model, different item);
- a background match;
- a box that's too loose.

Change `N_FAILURES` to see more.

In [ ]:
N_FAILURES = 3
default_cascade = Cascade(DinoRetriever(backbone, "crop_cls", CACHE_DIR), RETRIEVAL_K,
                          [SparseVerificationStage("sift", top_k=SPARSE_K, cache_dir=CACHE_DIR)], sync=sync)
default_cascade.prepare(items)
by_id = {it.item_id: it for it in items}
for query_id in worst_queries(default_report, n=N_FAILURES):
    display(Image(failure_grid(default_cascade, by_id[query_id], items, JUDGES["item"], k=5)))

## 8. Latency

This cell shows the time per query and per stage (median and p95, in ms) for every pipeline above. It excludes the offline indexing of the gallery.

**Read these as search cost, not as interactive latency.** In leave-one-out every query box is also a gallery box, so its DINOv2 description and keypoints were already computed while indexing. The numbers therefore leave out the query's own feature extraction:
- about 55 ms for DINOv2 at 448 px on an M2;
- about 20–50 ms for SIFT;
- about 15–110 ms for the learned matchers.

For the latency of a fresh query, see the timings line in `search.ipynb`.

The latency budget is ≤ 2 s per query on the GPU server and ≤ 5 s on the Mac (see `docs/log/`). Numbers measured on a different machine aren't comparable.

In [ ]:
latency = latency_table(stage1 + stage2)
latency

## 9. Save the results

This cell writes all tables of this run as CSV files to `results/<date-time>_<RUN_NAME>/` (gitignored), so runs can be compared later, for example after adding photos or changing a setting. Change `RUN_NAME` in the setup cell to label the run.

In [ ]:
tables = {"stage1": stage1_table, "stage2": stage2_table, "per_label": labels_table, "latency": latency}
if not grounding_table.empty:
    tables |= {"grounding": grounding_table, "localization": localization_table}
run_dir = save_tables(tables, RESULTS_DIR, RUN_NAME)
print("saved to", run_dir)